# Data Preprocessing

Fetch neuroimaging tabular data, clean it, and package it into PyTorch DataLoaders ready for training.


**About ABIDE** 

The Autism Brain Imaging Data Exchange (ABIDE) dataset provides previously collected resting state functional magnetic resonance imaging datasets from 539 individuals with ASD and 573 typical controls for the purpose of data sharing in the broader scientific community. 

Downloading the ABIDE dataset using Nilearn to extract the tabular Harvard-Oxford ROI time series (`derivatives=['rois_ho']`) alongside the phenotypic data, which contains key demographic covariates like Age and Sex.

In [1]:
# Import
import pandas as pd
import numpy as np
import nilearn
from nilearn.datasets import fetch_abide_pcp
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import os
import json


In [2]:
# ABIDE dataset
abide = fetch_abide_pcp(
    data_dir='../data',  # Look in our local data folder first
    derivatives=['rois_ho'],
    DX_GROUP=2)  # Healthy controls only
print(type(abide))

[fetch_abide_pcp] Dataset directory found: ../data/ABIDE_pcp


<class 'sklearn.utils._bunch.Bunch'>


## Clean and Merge Data

Creating a single, clean Pandas DataFrame containing our inputs and targets.

The phenotypic data metadata was retrieved from http://fcon_1000.projects.nitrc.org/indi/abide/abide_I.html, in the "ABIDE I Downloads" section, specifically the *Phenotypic Data Legend* download

From the phenotypic data, `AGE_AT_SCAN` and `SEX` are extracted and formated. 

In [3]:
print(abide.description)

.. _abide_dataset:

ABIDE PCP dataset

Access
------
See :func:`nilearn.datasets.fetch_abide_pcp`.

Notes
-----
The Autism Brain Imaging Data Exchange (ABIDE) dataset provides previously
collected resting state functional magnetic resonance imaging datasets
from 539 individuals with ASD and 573 typical controls for the purpose
of data sharing in the broader scientific community. This grass-root
initiative involved 16 international sites, sharing 20 samples yielding
1112 datasets composed of both MRI data and an extensive array of
phenotypic information common across nearly all sites (see below).

Note that this is the preprocessed version of ABIDE provided by the
preprocess connectome projects (PCP).

See :footcite:t:`Nielsen2013`.

Content
-------
    :'phenotypic': Behavioral information.

References
----------

.. footbibliography::

For more information about this dataset's structure:

http://preprocessed-connectomes-project.org/

https://childmind.org/science/global-open-science/h

In [4]:
#Explore abide
print(abide.keys())

# phenotypic data
df = pd.DataFrame(abide.phenotypic)
print(df.columns)
df.head(3)


dict_keys(['description', 'phenotypic', 'rois_ho'])
Index(['i', 'Unnamed: 0', 'SUB_ID', 'X', 'subject', 'SITE_ID', 'FILE_ID',
       'DX_GROUP', 'DSM_IV_TR', 'AGE_AT_SCAN',
       ...
       'qc_notes_rater_1', 'qc_anat_rater_2', 'qc_anat_notes_rater_2',
       'qc_func_rater_2', 'qc_func_notes_rater_2', 'qc_anat_rater_3',
       'qc_anat_notes_rater_3', 'qc_func_rater_3', 'qc_func_notes_rater_3',
       'SUB_IN_SMP'],
      dtype='object', length=106)


,i,Unnamed: 0,SUB_ID,X,subject,SITE_ID,FILE_ID,DX_GROUP,DSM_IV_TR,AGE_AT_SCAN,...,qc_notes_rater_1,qc_anat_rater_2,qc_anat_notes_rater_2,qc_func_rater_2,qc_func_notes_rater_2,qc_anat_rater_3,qc_anat_notes_rater_3,qc_func_rater_3,qc_func_notes_rater_3,SUB_IN_SMP
26,26,27,50030,27,50030,PITT,Pitt_0050030,2,0,25.12,...,NaN,OK,NaN,maybe,ic-parietal-cerebellum,OK,NaN,OK,NaN,1
27,27,28,50031,28,50031,PITT,Pitt_0050031,2,0,12.92,...,NaN,OK,NaN,maybe,ic-cerebellum_temporal_lobe,OK,NaN,OK,NaN,1
28,28,29,50032,29,50032,PITT,Pitt_0050032,2,0,19.80,...,NaN,OK,NaN,maybe,ic-cerebellum_temporal_lobe,OK,NaN,OK,NaN,1


In [5]:
# Attach the loaded ROI arrays directly to the dataframe
# (It turns out Nilearn already loaded the numbers)
df['ROI_DATA'] = abide.rois_ho

# Filter for Healthy Controls (DX_GROUP == 2)
df = df[df['DX_GROUP'] == 2].copy()

# Select Age, Sex, and our Data column
df = df[['AGE_AT_SCAN', 'SEX', 'ROI_DATA']].copy()

# Map SEX (ABIDE uses 1=Male, 2=Female) to 0 and 1
df['SEX'] = df['SEX'].map({1: 0, 2: 1})

print(df.dtypes)
df.head()

AGE_AT_SCAN    float64
SEX              int64
ROI_DATA        object
dtype: object


,AGE_AT_SCAN,SEX,ROI_DATA
26,25.12,0,"[[5.113387, 13.247622, 19.052804, 16.017521, 6..."
27,12.92,0,"[[33.056076, 29.351973, 69.177746, 65.114219, ..."
28,19.80,0,"[[9.342468, 5.513153, 13.548577, 0.008121, 9.7..."
29,12.15,0,"[[16.342057, 2.59721, 2.77128, -3.892353, -2.3..."
30,14.77,0,"[[-10.99951, -10.805331, 7.997344, 22.33833, -..."


The ABIDE dataset contains Resting-State fMRI (functional MRI) scans. Unlike a structural MRI which is just a single 3D static photo of the brain's anatomy, an fMRI takes a 3D picture of the brain every ~2 seconds for several minutes to track blood flow changes over time.

The data downloaded (rois_ho) has already been preprocessed. It took the fMRI data and grouped the brain into 111 Regions of Interest (ROIs) based on the Harvard-Oxford brain atlas.

So for one single patient, their roi_array is a 2D matrix where:

* Rows = Time points (e.g., 196 scans taken over time)
* Columns = The 111 brain regions

In [6]:
# Let's grab the very first patient's ROI array to see what it looks like
first_patient_roi = df['ROI_DATA'].iloc[0]

print("Original 2D array shape:", first_patient_roi.shape)
print(f" The patient has {first_patient_roi.shape[0]} time points, and {first_patient_roi.shape[1]} brain regions")

# Example: slice just the 1st region (column 0) across all time points
first_region_data = first_patient_roi[:, 0]
print("Shape of region 0 ([:, 0]):", first_region_data.shape)
print("First 5 time points for this region:", first_region_data[:5])

# Example: correlation between region 0 and region 1
first_fc = np.corrcoef(first_patient_roi[:, 0], first_patient_roi[:, 1])[0, 1]
print("FC between region 0 and region 1:", first_fc)
# This single number (Fisher z-transformed) is what we are appending to the target_fc

Original 2D array shape: (196, 111)
 The patient has 196 time points, and 111 brain regions
Shape of region 0 ([:, 0]): (196,)
First 5 time points for this region: [ 5.113387  2.274822 14.534917 -6.463246  0.504736]
FC between region 0 and region 1: 0.5461231753204139


The target variable is the pairwise Functional Connectivity (FC) between region 0 and region 1: the Pearson correlation of their time series, Fisher z-transformed (`np.arctanh`).

The mean of a single region is not used, since the fMRI time series are centered around zero during preprocessing.

By focusing on one specific pair of regions, the modeling objective mirrors a standard clinical growth chart, mapping demographics (Age and Sex) against a single biological metric.

In [7]:
# Loop through on the healthy files
target_fc = []
for roi_array in df['ROI_DATA']:
    # Correlation between region 0 and region 1
    r = np.corrcoef(roi_array[:, 0], roi_array[:, 1])[0, 1]
    # Fisher z-transform (clip to avoid infinite values at -1 or 1)
    target_fc.append(np.arctanh(np.clip(r, -0.9999, 0.9999)))

In [8]:
# Add the target column, drop the array column, and drop NaNs
df['FC'] = target_fc
df = df.drop(columns=['ROI_DATA'])
if df.isna().sum().sum() != 0:
    print("NA records", df.isna().sum())
    df = df.dropna()
else:
    pass

df.head()

,AGE_AT_SCAN,SEX,FC
26,25.12,0,0.612840
27,12.92,0,0.435302
28,19.80,0,0.537265
29,12.15,0,0.709478
30,14.77,0,0.759157


**Feature Scaling**

Z-score scaling of `AGE_AT_SCAN` and `FC`. The scaling parameters are saved to transform predictions back to original units.

In [9]:
# Mean and std
age_mean = float(df['AGE_AT_SCAN'].mean())
age_std = float(df['AGE_AT_SCAN'].std())
fc_mean = float(df['FC'].mean())
fc_std = float(df['FC'].std())

# Scaled columns
df['Age_scaled'] = (df['AGE_AT_SCAN'] - age_mean) / age_std
df['FC_scaled'] = (df['FC'] - fc_mean) / fc_std

# Save scaling parameters
scaling_params = {'age_mean': age_mean, 'age_std': age_std, 'fc_mean': fc_mean, 'fc_std': fc_std}
os.makedirs("../data/processed", exist_ok=True)
with open("../data/processed/scaling_params.json", "w") as f:
    json.dump(scaling_params, f, indent=4)

print(scaling_params)
df.head()

{'age_mean': 16.83903311965812, 'age_std': 7.242265848499033, 'fc_mean': 0.5217374600625769, 'fc_std': 0.1984266098918636}


,AGE_AT_SCAN,SEX,FC,Age_scaled,FC_scaled
26,25.12,0,0.612840,1.143422,0.459125
27,12.92,0,0.435302,-0.541134,-0.435606
28,19.80,0,0.537265,0.408845,0.078256
29,12.15,0,0.709478,-0.647454,0.946148
30,14.77,0,0.759157,-0.285689,1.196512


# PyTorch Dataset

Wrapping the dataframe in a custom PyTorch `Dataset` class. 

The `__init__` method isolates the demographic input features (`X`) and the target FC (`y`), converting them into PyTorch  tensors.

In [10]:
class BrainNormativeDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, feature_cols=['Age_scaled', 'SEX'], target_col='FC_scaled'):
        """
        Custom PyTorch Dataset for normative brain centile estimation
        """
        # Convert inputs and targets to tensors
        self.X = torch.tensor(dataframe[feature_cols].values, dtype=torch.float32)
        self.y = torch.tensor(dataframe[target_col].values, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# call dataset
dataset = BrainNormativeDataset(df)
print(f"Total samples in dataset: {len(dataset)}")
print("The first person (record): ")
print(f"[Age,Sex]: {dataset[0][0]}") # For sex, 0 = MALE, and 1 = FEMALE
print(f"[FC]: {dataset[0][1]}")


Total samples in dataset: 468
The first person (record): 


[Age,Sex]: tensor([1.1434, 0.0000])
[FC]: tensor([0.4591])


**DataLoader**

Splitting the data into training (70%), validation (15%) and testing (15%) sets.

In [11]:
# Set random seed for reproducible train/val/test split
generator = torch.Generator().manual_seed(42)

# Split data
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=generator)

print(f"Training set size: {len(train_dataset)} samples")
print(f"Validation set size: {len(val_dataset)} samples")
print(f"Test set size: {len(test_dataset)} samples")

# dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of train batches: {len(train_loader)} (batch_size={batch_size})")
print(f"Number of validation batches: {len(val_loader)} (batch_size={batch_size})")
print(f"Number of test batches: {len(test_loader)} (batch_size={batch_size})")

# inspect the first batch from train_loader
for batch_X, batch_y in train_loader:
    print(f"Batch X shape: {batch_X.shape}")
    print(f"Batch y shape: {batch_y.shape}")
    print(f"First 3 samples [Age, Sex]:\n{batch_X[:3]}")
    print(f"First 3 samples [FC]:\n{batch_y[:3]}")

    break


Training set size: 327 samples
Validation set size: 70 samples
Test set size: 71 samples
Number of train batches: 21 (batch_size=16)
Number of validation batches: 5 (batch_size=16)
Number of test batches: 5 (batch_size=16)
Batch X shape: torch.Size([16, 2])
Batch y shape: torch.Size([16, 1])
First 3 samples [Age, Sex]:
tensor([[-0.3879,  0.0000],
        [ 0.6546,  1.0000],
        [-0.3837,  1.0000]])
First 3 samples [FC]:
tensor([[ 1.6420],
        [-0.1848],
        [ 0.8093]])


In [12]:
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/abide_normative_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"Dataset saved at: {output_path}")


Dataset saved at: ../data/processed/abide_normative_cleaned.csv
